In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import warnings
import lightgbm as lgb 
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
import optuna

c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FEATURES_FILE = "../datasets/features.csv"
LABEL_FILE    = "../datasets/risk_label.csv"
MODEL_DIR     = "../models/"
OUT_SCORES    = "../datasets/all_area_scores.csv"

os.makedirs(MODEL_DIR, exist_ok=True)
 
N_FOLDS         = 3     
OPTUNA_TRIALS   = 100
RANDOM_STATE    = 42

In [3]:
features_df = pd.read_csv(FEATURES_FILE)
label_df    = pd.read_csv(LABEL_FILE)
 
df = features_df.merge(label_df, on="boundary_POL_STN_NM", how="left")
 
df_train   = df[df["risk_score"].notna()].copy().reset_index(drop=True)
df_predict = df[df["risk_score"].isna()].copy().reset_index(drop=True)

In [4]:
EXCLUDE = {"boundary_POL_STN_NM", "risk_score"}
BASE_FEATURES = [
    c for c in features_df.columns
    if c not in EXCLUDE
    and features_df[c].dtype in [np.float64, np.int64, float, int]
]

In [5]:
def engineer_features(df: pd.DataFrame,
                      fit_df: pd.DataFrame = None) -> pd.DataFrame:

    df = df.copy()
    ref = fit_df if fit_df is not None else df
 
    df["violent_crime_ratio"] = (
        (df["murder_count"] + df["rape_count"] +
         df["robbery_count"] + df["assault_count"])
        / (df["murder_count"] + df["rape_count"] + df["robbery_count"] +
           df["assault_count"] + df["sexual_harassment_count"] +
           df["gangrape_count"] + df["theft_count"] + 1e-9)
    )
 

    df["severity_crime_score"] = (
        df["murder_count"]             * 10.0 +
        df["rape_count"]               *  8.0 +
        df["gangrape_count"]           *  8.0 +
        df["robbery_count"]            *  5.0 +
        df["assault_count"]            *  4.0 +
        df["sexual_harassment_count"]  *  3.0 +
        df["theft_count"]              *  1.0
    )
 
    df["fatal_accident_ratio"] = (
        df["fatal_accidents_avg"]
        / (df["fatal_accidents_avg"] + df["injury_accidents_avg"] + 1e-9)
    )
 
    df["metro_accessibility"]    = 1.0 / (df["nearest_metro_km"]    + 0.5)
    df["hospital_accessibility"] = 1.0 / (df["nearest_hospital_km"] + 0.5)
    df["fire_accessibility"]     = 1.0 / (df["nearest_fire_stn_km"] + 0.5)
 
    df["emergency_access_score"] = (
        df["metro_accessibility"] +
        df["hospital_accessibility"] +
        df["fire_accessibility"]
    ) / 3.0
 
    df["safety_perception_score"] = (
        df["lighting_score"]     * 0.30 +
        df["security_score"]     * 0.25 +
        df["people_score"]       * 0.20 +
        df["gender_usage_score"] * 0.15 +
        df["walkpath_score"]     * 0.10
    )
 
    rape_max       = ref["rape_count"].max() + 1e-9
    harassment_max = ref["sexual_harassment_count"].max() + 1e-9
 
    df["women_safety_score"] = (
        (1 - df["rape_count"]              / rape_max)       * 0.40 +
        (1 - df["sexual_harassment_count"] / harassment_max) * 0.35 +
        (df["gender_usage_score"] / 5.0)                     * 0.25
    )

    df["tourist_exposure"] = (
        np.log1p(df["tourist_poi_count"]) * 0.6 +
        df["metro_accessibility"]          * 0.4
    )

    df["crime_x_darkness"] = (
        df["severity_crime_score"] * (5.0 - df["lighting_score"])
    )
 
    return df

In [6]:
df_train_eng   = engineer_features(df_train,   fit_df=df_train)
df_predict_eng = engineer_features(df_predict, fit_df=df_train)
 
ALL_FEATURES = [
    c for c in df_train_eng.columns
    if c not in EXCLUDE
    and df_train_eng[c].dtype in [np.float64, np.int64, float, int]
]
 
print(f"Engineered features: {len(ALL_FEATURES)}")
 
X_full = df_train_eng[ALL_FEATURES].values
y      = df_train["risk_score"].values
groups = df_train["boundary_POL_STN_NM"].values

Engineered features: 43


In [7]:
df_train_eng.columns

Index(['boundary_POL_STN_NM', 'murder_count', 'rape_count', 'robbery_count',
       'theft_count', 'assault_count', 'sexual_harassment_count',
       'gangrape_count', 'crime_yoy_delta', 'area', 'lighting_score',
       'visibility_score', 'people_score', 'security_score', 'transport_score',
       'walkpath_score', 'feeling_score', 'gender_usage_score',
       'bus_stop_count', 'hospital_count', 'tourist_poi_count',
       'public_toilet_count', 'nearest_metro_km', 'nearest_hospital_km',
       'nearest_fire_stn_km', 'nearest_poi_km', 'nearest_police_stn_km',
       'fatal_accidents_avg', 'injury_accidents_avg', 'avg_sentiment_score',
       'pct_negative_reviews', 'neg_keyword_rate', 'avg_star_rating',
       'risk_score', 'violent_crime_ratio', 'severity_crime_score',
       'fatal_accident_ratio', 'metro_accessibility', 'hospital_accessibility',
       'fire_accessibility', 'emergency_access_score',
       'safety_perception_score', 'women_safety_score', 'tourist_exposure',
       

In [8]:
df_train_eng.drop(columns = ['boundary_POL_STN_NM']).corr()

,murder_count,rape_count,robbery_count,theft_count,assault_count,sexual_harassment_count,gangrape_count,crime_yoy_delta,area,lighting_score,...,severity_crime_score,fatal_accident_ratio,metro_accessibility,hospital_accessibility,fire_accessibility,emergency_access_score,safety_perception_score,women_safety_score,tourist_exposure,crime_x_darkness
murder_count,1.000000,0.450616,0.480931,0.230316,0.315104,0.317542,0.183770,NaN,NaN,0.049375,...,0.344966,0.093131,0.039682,0.093058,0.098820,0.101726,0.011256,-0.478176,-0.041601,0.328061
rape_count,0.450616,1.000000,0.334776,0.320446,0.606808,0.302625,0.394933,NaN,NaN,-0.095329,...,0.464308,0.050333,0.002439,0.029305,0.080831,0.046091,-0.054472,-0.829454,-0.082518,0.477696
robbery_count,0.480931,0.334776,1.000000,0.638089,0.304856,0.472956,0.136183,NaN,NaN,-0.096298,...,0.701629,0.157941,-0.108082,-0.020321,-0.109982,-0.098345,-0.074077,-0.491928,-0.031147,0.719842
theft_count,0.230316,0.320446,0.638089,1.000000,0.475083,0.336533,0.145051,NaN,NaN,0.011581,...,0.982673,0.144180,0.016647,0.154445,-0.018439,0.079698,-0.007727,-0.409107,0.071144,0.953802
assault_count,0.315104,0.606808,0.304856,0.475083,1.000000,0.379746,0.269539,NaN,NaN,-0.016358,...,0.576566,0.118388,0.054921,0.180583,0.030571,0.127980,0.011034,-0.614127,0.001993,0.553633
sexual_harassment_count,0.317542,0.302625,0.472956,0.336533,0.379746,1.000000,0.150508,NaN,NaN,-0.021156,...,0.411951,0.090658,0.004179,0.179065,0.011337,0.097947,0.011489,-0.770251,-0.032047,0.404531
gangrape_count,0.183770,0.394933,0.136183,0.145051,0.269539,0.150508,1.000000,NaN,NaN,-0.102362,...,0.215171,-0.082735,0.008912,-0.017805,-0.047577,-0.022904,0.025300,-0.336171,-0.120008,0.242165
crime_yoy_delta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lighting_score,0.049375,-0.095329,-0.096298,0.011581,-0.016358,-0.021156,-0.102362,NaN,NaN,1.000000,...,-0.007231,0.279428,0.323964,0.255457,0.354640,0.403911,0.783612,0.104606,0.095628,-0.204283


In [9]:
def xgb_oof_rmse(params: dict, X: np.ndarray, y: np.ndarray,
                 groups: np.ndarray, df_for_eng: pd.DataFrame) -> float:
    gkf      = GroupKFold(n_splits=N_FOLDS)
    oof      = np.zeros(len(y))
    feat_idx = None   # computed once from first fold
 
    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        fold_train_df = df_for_eng.iloc[train_idx].copy()
        fold_val_df   = df_for_eng.iloc[val_idx].copy()
 
        fold_train_eng = engineer_features(fold_train_df, fit_df=fold_train_df)
        fold_val_eng   = engineer_features(fold_val_df,   fit_df=fold_train_df)
 
        fold_feats = [
            c for c in fold_train_eng.columns
            if c not in EXCLUDE
            and fold_train_eng[c].dtype in [np.float64, np.int64, float, int]
        ]
 
        X_tr = fold_train_eng[fold_feats].values
        X_vl = fold_val_eng[fold_feats].values
        y_tr = y[train_idx]
        y_vl = y[val_idx]
 
        model = XGBRegressor(
            **params,
            early_stopping_rounds=30,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
        preds = np.clip(model.predict(X_vl), 0, 100)
        oof[val_idx] = preds
 
    return float(np.sqrt(mean_squared_error(y, oof)))

In [10]:
def objective(trial):
    params = {
        "n_estimators":    trial.suggest_int("n_estimators", 200, 800),
        "max_depth":       trial.suggest_int("max_depth", 3, 6),
        "learning_rate":   trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample":       trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight":trial.suggest_int("min_child_weight", 2, 10),
        "reg_alpha":       trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda":      trial.suggest_float("reg_lambda", 0.3, 3.0),
    }
    return xgb_oof_rmse(params, X_full, y, groups, df_train)

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

BEST_XGB_PARAMS = study.best_params
print(f"\nBest OOF RMSE: {study.best_value:.4f}")
print(f"Best params:   {json.dumps(BEST_XGB_PARAMS, indent=2)}")

[I 2026-04-05 15:26:28,822] A new study created in memory with name: no-name-f86fa062-d903-485e-8666-40da064a8905
Best trial: 0. Best value: 6.65388:   1%|          | 1/100 [00:00<01:05,  1.52it/s]

[I 2026-04-05 15:26:29,481] Trial 0 finished with value: 6.653882254921881 and parameters: {'n_estimators': 425, 'max_depth': 6, 'learning_rate': 0.05395030966670229, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.4936111842654619, 'min_child_weight': 3, 'reg_alpha': 0.11616722433639892, 'reg_lambda': 2.638675593592325}. Best is trial 0 with value: 6.653882254921881.


Best trial: 0. Best value: 6.65388:   2%|▏         | 2/100 [00:01<01:07,  1.45it/s]

[I 2026-04-05 15:26:30,191] Trial 1 finished with value: 7.470142771233015 and parameters: {'n_estimators': 561, 'max_depth': 5, 'learning_rate': 0.010485387725194618, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.899465584480253, 'min_child_weight': 3, 'reg_alpha': 0.36364993441420124, 'reg_lambda': 0.7951921766042713}. Best is trial 0 with value: 6.653882254921881.


Best trial: 2. Best value: 6.5493:   3%|▎         | 3/100 [00:01<00:50,  1.94it/s] 

...

[I 2026-04-05 15:26:52,842] Trial 94 finished with value: 6.159231733477471 and parameters: {'n_estimators': 727, 'max_depth': 3, 'learning_rate': 0.08961105193296537, 'subsample': 0.6009665652192777, 'colsample_bytree': 0.42562611483859303, 'min_child_weight': 6, 'reg_alpha': 0.3171618710668505, 'reg_lambda': 0.3092639302096971}. Best is trial 84 with value: 6.0750615723931976.
[I 2026-04-05 15:26:52,982] Trial 95 finished with value: 6.323546226672966 and parameters: {'n_estimators': 713, 'max_depth': 3, 'learning_rate': 0.07130825782539751, 'subsample': 0.6029498530976589, 'colsample_bytree': 0.4390703792085757, 'min_child_weight': 6, 'reg_alpha': 0.3412649867961549, 'reg_lambda': 0.7763791207637158}. Best is trial 84 with value: 6.0750615723931976.


Best trial: 84. Best value: 6.07506:  98%|█████████▊| 98/100 [00:24<00:00,  6.80it/s]

[I 2026-04-05 15:26:53,134] Trial 96 finished with value: 6.386568901688414 and parameters: {'n_estimators': 724, 'max_depth': 3, 'learning_rate': 0.08719059671770323, 'subsample': 0.6197777582388639, 'colsample_bytree': 0.49504653412858884, 'min_child_weight': 5, 'reg_alpha': 0.31894671712643274, 'reg_lambda': 0.33516765332863113}. Best is trial 84 with value: 6.0750615723931976.
[I 2026-04-05 15:26:53,268] Trial 97 finished with value: 6.390792298764853 and parameters: {'n_estimators': 690, 'max_depth': 3, 'learning_rate': 0.07433711301154798, 'subsample': 0.551570269638433, 'colsample_bytree': 0.41544004475360286, 'min_child_weight': 6, 'reg_alpha': 0.11679778223509518, 'reg_lambda': 0.3736441532131137}. Best is trial 84 with value: 6.0750615723931976.


Best trial: 84. Best value: 6.07506: 100%|██████████| 100/100 [00:24<00:00,  4.05it/s]

[I 2026-04-05 15:26:53,431] Trial 98 finished with value: 6.464621848092327 and parameters: {'n_estimators': 709, 'max_depth': 3, 'learning_rate': 0.09640181687731485, 'subsample': 0.5280486155648068, 'colsample_bytree': 0.9015384363083719, 'min_child_weight': 5, 'reg_alpha': 0.2316661631706101, 'reg_lambda': 0.8459712971597444}. Best is trial 84 with value: 6.0750615723931976.
[I 2026-04-05 15:26:53,542] Trial 99 finished with value: 6.484945337476047 and parameters: {'n_estimators': 766, 'max_depth': 3, 'learning_rate': 0.08233185249298856, 'subsample': 0.8424022903182253, 'colsample_bytree': 0.4717901293645276, 'min_child_weight': 7, 'reg_alpha': 0.16018234494500294, 'reg_lambda': 0.30886309779710897}. Best is trial 84 with value: 6.0750615723931976.

Best OOF RMSE: 6.0751
Best params:   {
  "n_estimators": 752,
  "max_depth": 3,
  "learning_rate": 0.07374833153857586,
  "subsample": 0.5905452227234542,
  "colsample_bytree": 0.4326287745994814,
  "min_child_weight": 5,
  "reg_alpha"

In [11]:
gkf = GroupKFold(n_splits=N_FOLDS)
oof_xgb  = np.zeros(len(df_train))
oof_lgb  = np.zeros(len(df_train)) 
fold_scores = []

In [12]:
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X_full, y, groups=groups), 1
):
    fold_train_df  = df_train.iloc[train_idx].copy()
    fold_val_df    = df_train.iloc[val_idx].copy()
    fold_train_eng = engineer_features(fold_train_df, fit_df=fold_train_df)
    fold_val_eng   = engineer_features(fold_val_df,   fit_df=fold_train_df)
 
    fold_feats = [
        c for c in fold_train_eng.columns
        if c not in EXCLUDE
        and fold_train_eng[c].dtype in [np.float64, np.int64, float, int]
    ]
 
    X_tr = fold_train_eng[fold_feats].values
    X_vl = fold_val_eng[fold_feats].values
    y_tr = y[train_idx]
    y_vl = y[val_idx]
 
    xgb_model = XGBRegressor(
        **BEST_XGB_PARAMS,
        early_stopping_rounds=30,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    xgb_preds = np.clip(xgb_model.predict(X_vl), 0, 100)
    oof_xgb[val_idx] = xgb_preds

    lgb_params = {
        "n_estimators":     BEST_XGB_PARAMS.get("n_estimators", 500),
        "max_depth":        BEST_XGB_PARAMS.get("max_depth", 4),
        "learning_rate":    BEST_XGB_PARAMS.get("learning_rate", 0.03),
        "subsample":        BEST_XGB_PARAMS.get("subsample", 0.8),
        "colsample_bytree": BEST_XGB_PARAMS.get("colsample_bytree", 0.8),
        "min_child_samples":max(5, BEST_XGB_PARAMS.get("min_child_weight", 3)),
        "reg_alpha":        BEST_XGB_PARAMS.get("reg_alpha", 0.1),
        "reg_lambda":       BEST_XGB_PARAMS.get("reg_lambda", 1.5),
        "random_state":     RANDOM_STATE,
        "n_jobs":           -1,
        "verbose":          -1,
    }
    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(-1)],
    )
    lgb_preds = np.clip(lgb_model.predict(X_vl), 0, 100)
    oof_lgb[val_idx] = lgb_preds
    ensemble_preds = 0.5 * xgb_preds + 0.5 * lgb_preds

    fold_r2   = r2_score(y_vl, ensemble_preds)
    fold_rmse = np.sqrt(mean_squared_error(y_vl, ensemble_preds))
    fold_mae  = mean_absolute_error(y_vl, ensemble_preds)
    fold_scores.append({"r2": fold_r2, "rmse": fold_rmse, "mae": fold_mae})
 
    val_areas = groups[val_idx]


c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [13]:
oof_ensemble = 0.6 * oof_xgb + 0.4 * oof_lgb
oof_r2   = r2_score(y, oof_ensemble)
oof_rmse = np.sqrt(mean_squared_error(y, oof_ensemble))
oof_mae  = mean_absolute_error(y, oof_ensemble)

In [14]:
def categorize(scores):
    LOW_THRESH = np.percentile(scores, 50)   
    HIGH_THRESH = np.percentile(scores, 85)  
    return pd.cut(scores, bins=[-np.inf, LOW_THRESH, HIGH_THRESH, np.inf],
                  labels=["Low", "Medium", "High"])
 
actual_cat    = categorize(y)
predicted_cat = categorize(oof_ensemble)
cat_acc       = (actual_cat == predicted_cat).mean()
critical_err  = ((actual_cat == "High") & (predicted_cat == "Low")).sum()
 
print(f"Category accuracy : {cat_acc:.3f}")
print(f"Critical errors   : {critical_err}  (High predicted as Low)")
 
print("\nConfusion matrix:")
print(pd.crosstab(actual_cat, predicted_cat,
                  rownames=["Actual"], colnames=["Predicted"]))
 
# Residual analysis
residuals = y - oof_ensemble
error_df = pd.DataFrame({
    "area":      groups,
    "actual":    y,
    "predicted": oof_ensemble,
    "error":     residuals,
    "abs_error": np.abs(residuals),
}).sort_values("abs_error", ascending=False)

print("\nTop 10 worst predictions:")
print(error_df.head(10)[["area","actual","predicted","error"]].to_string(index=False))

Category accuracy : 0.745
Critical errors   : 1  (High predicted as Low)

Confusion matrix:
Predicted  Low  Medium  High
Actual                      
Low         66      12     1
Medium      12      35     7
High         1       7    16

Top 10 worst predictions:
              area  actual  predicted      error
  PS SUBHASH PLACE   65.95  41.703639  24.246361
PS SUNLIGHT COLONY   58.17  38.212752  19.957248
     PS JAGAT PURI   55.86  39.424498  16.435502
     PS ROOP NAGAR   42.34  26.084783  16.255217
    PS BHAJAN PURA   14.51  30.083704 -15.573704
    PS GOVIND PURI   20.71  35.805951 -15.095951
  PS MALVIYA NAGAR   48.52  34.514401  14.005599
    PS MANDIR MARG   23.48  36.202547 -12.722547
      PS NEB SARAI   25.22  37.767514 -12.547514
    PS KALYAN PURI   31.96  19.925003  12.034997


In [15]:
df_train_final = engineer_features(df_train, fit_df=df_train)
final_feats = [
    c for c in df_train_final.columns
    if c not in EXCLUDE
    and df_train_final[c].dtype in [np.float64, np.int64, float, int]
]
X_train_final = df_train_final[final_feats].values
 
final_xgb = XGBRegressor(
    **{k: v for k, v in BEST_XGB_PARAMS.items()},
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
)
final_xgb.fit(X_train_final, y)

lgb_final_params = {
        k: (max(5, v) if k == "min_child_weight" else v)
        for k, v in BEST_XGB_PARAMS.items()
    }
lgb_final_params["min_child_samples"] = lgb_final_params.pop(
    "min_child_weight", 5)
lgb_final_params.update({
    "random_state": RANDOM_STATE, "n_jobs": -1, "verbose": -1
})
final_lgb = lgb.LGBMRegressor(**lgb_final_params)
final_lgb.fit(X_train_final, y)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,3
,learning_rate,0.07374833153857586
,n_estimators,752
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [16]:
importance_df = pd.DataFrame({
    "feature":    final_feats,
    "importance": final_xgb.feature_importances_,
}).sort_values("importance", ascending=False)

zero_imp = importance_df[importance_df["importance"] < 0.005]

if len(zero_imp):
    for _, row in zero_imp.iterrows():
        print(f"  {row['feature']:<35} {row['importance']:.4f}")

  nearest_metro_km                    0.0049
  avg_sentiment_score                 0.0047
  neg_keyword_rate                    0.0046
  gender_usage_score                  0.0046
  fire_accessibility                  0.0044
  hospital_count                      0.0031
  nearest_police_stn_km               0.0030
  metro_accessibility                 0.0026
  public_toilet_count                 0.0024
  crime_yoy_delta                     0.0000
  area                                0.0000


In [17]:
selected_features = [
    c for c in final_feats
    if c not in zero_imp["feature"].values
]
print(f"Selected features after importance pruning: {len(selected_features)}")

Selected features after importance pruning: 32


In [18]:
X_new = df_train_eng[selected_features].values
print(f"X_new shape: {X_new.shape}")

X_new shape: (157, 32)


In [19]:
oof_xgb  = np.zeros(len(df_train))
oof_lgb  = np.zeros(len(df_train))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X_new, y, groups=groups), 1
):
    fold_train_df  = df_train.iloc[train_idx].copy()
    fold_val_df    = df_train.iloc[val_idx].copy()
    fold_train_eng = engineer_features(fold_train_df, fit_df=fold_train_df)
    fold_val_eng   = engineer_features(fold_val_df,   fit_df=fold_train_df)
 
    fold_feats = [c for c in selected_features if c in fold_train_eng.columns]
 
    X_tr = fold_train_eng[fold_feats].values
    X_vl = fold_val_eng[fold_feats].values
    y_tr = y[train_idx]
    y_vl = y[val_idx]
 
    xgb_model = XGBRegressor(
        **BEST_XGB_PARAMS,
        early_stopping_rounds=30,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    xgb_preds = np.clip(xgb_model.predict(X_vl), 0, 100)
    oof_xgb[val_idx] = xgb_preds

    lgb_params = {
        "n_estimators":     BEST_XGB_PARAMS.get("n_estimators", 500),
        "max_depth":        BEST_XGB_PARAMS.get("max_depth", 4),
        "learning_rate":    BEST_XGB_PARAMS.get("learning_rate", 0.03),
        "subsample":        BEST_XGB_PARAMS.get("subsample", 0.8),
        "colsample_bytree": BEST_XGB_PARAMS.get("colsample_bytree", 0.8),
        "min_child_samples":max(5, BEST_XGB_PARAMS.get("min_child_weight", 3)),
        "reg_alpha":        BEST_XGB_PARAMS.get("reg_alpha", 0.1),
        "reg_lambda":       BEST_XGB_PARAMS.get("reg_lambda", 1.5),
        "random_state":     RANDOM_STATE,
        "n_jobs":           -1,
        "verbose":          -1,
    }
    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(-1)],
    )
    lgb_preds = np.clip(lgb_model.predict(X_vl), 0, 100)
    oof_lgb[val_idx] = lgb_preds
    ensemble_preds = 0.6 * xgb_preds + 0.4 * lgb_preds

    fold_r2   = r2_score(y_vl, ensemble_preds)
    fold_rmse = np.sqrt(mean_squared_error(y_vl, ensemble_preds))
    fold_mae  = mean_absolute_error(y_vl, ensemble_preds)
    fold_scores.append({"r2": fold_r2, "rmse": fold_rmse, "mae": fold_mae})

oof_ensemble = 0.6 * oof_xgb + 0.4 * oof_lgb
oof_r2   = r2_score(y, oof_ensemble)
oof_rmse = np.sqrt(mean_squared_error(y, oof_ensemble))
oof_mae  = mean_absolute_error(y, oof_ensemble)

actual_cat    = categorize(y)
predicted_cat = categorize(oof_ensemble)
cat_acc       = (actual_cat == predicted_cat).mean()
critical_err  = ((actual_cat == "High") & (predicted_cat == "Low")).sum()

print(f"Selected-feature OOF -> R2: {oof_r2:.4f}, RMSE: {oof_rmse:.4f}, MAE: {oof_mae:.4f}")
print(f"Category accuracy : {cat_acc:.3f}")
print(f"Critical errors   : {critical_err}  (High predicted as Low)")

c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Selected-feature OOF -> R2: 0.6589, RMSE: 6.2045, MAE: 4.6825
Category accuracy : 0.694
Critical errors   : 1  (High predicted as Low)


c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [20]:
df_train_final = engineer_features(df_train, fit_df=df_train)
final_feats = selected_features.copy()
X_train_final = df_train_final[final_feats].values
 
final_xgb = XGBRegressor(
    **{k: v for k, v in BEST_XGB_PARAMS.items()},
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
 )
final_xgb.fit(X_train_final, y)

lgb_final_params = {
        k: (max(5, v) if k == "min_child_weight" else v)
        for k, v in BEST_XGB_PARAMS.items()
    }
lgb_final_params["min_child_samples"] = lgb_final_params.pop(
    "min_child_weight", 5)
lgb_final_params.update({
    "random_state": RANDOM_STATE, "n_jobs": -1, "verbose": -1
})
final_lgb = lgb.LGBMRegressor(**lgb_final_params)
final_lgb.fit(X_train_final, y)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,3
,learning_rate,0.07374833153857586
,n_estimators,752
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [21]:
X_train_final.shape

(157, 32)

In [22]:
train_scores = pd.DataFrame({
    "boundary_POL_STN_NM": groups,
    "base_score":          np.clip(oof_ensemble, 0, 100).round(2),
    "source":              "oof_prediction",
})
 
# Predict-only rows
if len(df_predict) > 0:
    df_pred_eng   = engineer_features(df_predict, fit_df=df_train)
    X_pred_final  = df_pred_eng[final_feats].values
 
    xgb_pred_scores = np.clip(final_xgb.predict(X_pred_final), 0, 100)
    lgb_pred_scores = np.clip(final_lgb.predict(X_pred_final), 0, 100)
    pred_scores = 0.6 * xgb_pred_scores + 0.4 * lgb_pred_scores

 
    pred_df = pd.DataFrame({
        "boundary_POL_STN_NM": df_predict["boundary_POL_STN_NM"].values,
        "base_score":          pred_scores.round(2),
        "source":              "model_prediction",
    })
else:
    pred_df = pd.DataFrame()

all_scores = pd.concat([train_scores, pred_df], ignore_index=True)
all_scores["risk_category"] = categorize(all_scores["base_score"])
all_scores = all_scores.sort_values("base_score", ascending=False)
all_scores.to_csv(OUT_SCORES, index=False)
 
print(f"\nRisk distribution (all 180 areas):")
print(all_scores["risk_category"].value_counts().to_string())
print(f"\nTop 10 highest risk areas:")
print(all_scores.head(10)[
    ["boundary_POL_STN_NM","base_score","risk_category","source"]
].to_string(index=False))


Risk distribution (all 180 areas):
risk_category
Low       90
Medium    63
High      27

Top 10 highest risk areas:
boundary_POL_STN_NM  base_score risk_category         source
  PS PARSHANT VIHAR       49.33          High oof_prediction
          PS BAWANA       46.31          High oof_prediction
    PS PUNJABI BAGH       45.82          High oof_prediction
      PS MOTI NAGAR       45.31          High oof_prediction
   PS MAHENDRA PARK       43.35          High oof_prediction
       PS RANI BAGH       43.09          High oof_prediction
   PS SUBHASH PLACE       42.73          High oof_prediction
  PS RAJOURI GARDEN       42.64          High oof_prediction
      PS NAND NAGRI       42.59          High oof_prediction
         PS NANGLOI       40.45          High oof_prediction


c:\TRJ\programming\projects\ml\sentry\Sentry_app\ml\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [23]:
joblib.dump(final_xgb, os.path.join(MODEL_DIR, "risk_model_xgb.pkl"))
joblib.dump(final_lgb, os.path.join(MODEL_DIR, "risk_model_lgb.pkl"))
 
metadata = {
    "version":            "v1",
    "feature_names":      final_feats,
    "base_features":      BASE_FEATURES,
    "oof_r2":             round(oof_r2,   4),
    "oof_rmse":           round(oof_rmse, 4),
    "oof_mae":            round(oof_mae,  4),
    "category_accuracy":  round(float(cat_acc), 4),
    "critical_errors":    int(critical_err),
    "n_training_areas":   len(df_train),
    "n_folds":            N_FOLDS,
    "ensemble":           "XGBoost+LightGBM (0.6/0.4)",
    "best_xgb_params":    BEST_XGB_PARAMS,
    "optuna_trials":      OPTUNA_TRIALS,
}
with open(os.path.join(MODEL_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
 
importance_df.to_csv(
    os.path.join(MODEL_DIR, "feature_importance.csv"), index=False)